In [1]:
import torch
import torch.nn as nn


class Generator(nn.Module):
    """
    Takes a random latent noise vector z and transforms it into an artificial image.
    """

    def __init__(self, latent_dim=100, img_shape=(1, 28, 28)):
        super().__init__()
        self.img_shape = img_shape

        self.model = nn.Sequential(
            # Input: [Batch, Latent_Dim]
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            # Map out to flattened pixel space dimensions
            nn.Linear(1024, int(torch.prod(torch.tensor(img_shape)))),
            # Tanh bounds output pixel intensities between -1.0 and 1.0
            nn.Tanh(),
        )

    def forward(self, z):
        img_flat = self.model(z)
        # Reshape flat vector into spatial image dimensions: [Batch, C, H, W]
        return img_flat.view(img_flat.size(0), *self.img_shape)


class Discriminator(nn.Module):
    """
    Takes an image (real or fake) and outputs a scalar probability score
    indicating whether the input is a genuine data sample.
    """

    def __init__(self, img_shape=(1, 28, 28)):
        super().__init__()

        self.model = nn.Sequential(
            # Flatten image into a continuous feature vector
            nn.Linear(int(torch.prod(torch.tensor(img_shape))), 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            # Sigmoid outputs a probability score strictly between 0.0 and 1.0
            nn.Sigmoid(),
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity


In [2]:
def train_gan_step(
    generator,
    discriminator,
    real_images,
    latent_dim,
    g_optimizer,
    d_optimizer,
    loss_fn,
    device,
):
    """
    Executes a single operational optimization step for both networks.
    """
    batch_size = real_images.size(0)
    real_images = real_images.to(device)

    # Ground truth reference tensors for Binary Cross-Entropy
    real_labels = torch.ones(batch_size, 1).to(device)
    fake_labels = torch.zeros(batch_size, 1).to(device)

    # -----------------------------------------------------------------
    #  Phase 1: Train Discriminator
    # -----------------------------------------------------------------
    d_optimizer.zero_grad()

    # Measure ability to identify real imagery
    outputs_real = discriminator(real_images)
    d_loss_real = loss_fn(outputs_real, real_labels)

    # Generate fake images from random noise
    z = torch.randn(batch_size, latent_dim).to(device)
    fake_images = generator(z)

    # Measure ability to intercept generated synthetic imagery
    # (.detach() cuts off generator gradients during discriminator pass)
    outputs_fake = discriminator(fake_images.detach())
    d_loss_fake = loss_fn(outputs_fake, fake_labels)

    # Backpropagate combined discriminator error
    d_loss = d_loss_real + d_loss_fake
    d_loss.backward()
    d_optimizer.step()

    # -----------------------------------------------------------------
    #  Phase 2: Train Generator
    # -----------------------------------------------------------------
    g_optimizer.zero_grad()

    # Re-evaluate fake images without truncating graph paths
    outputs_fake_for_g = discriminator(fake_images)

    # The Generator wins if the Discriminator mistakes fake images for real ones
    g_loss = loss_fn(outputs_fake_for_g, real_labels)

    g_loss.backward()
    g_optimizer.step()

    return d_loss.item(), g_loss.item()


In [4]:
if __name__ == "__main__":
    # Setup execution configurations
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    LATENT_DIM = 100
    BATCH_SIZE = 16
    IMG_SHAPE = (1, 28, 28)

    # Instantiate Networks
    gen = Generator(latent_dim=LATENT_DIM, img_shape=IMG_SHAPE).to(device)
    disc = Discriminator(img_shape=IMG_SHAPE).to(device)

    # Hyperparameters & Optimizers
    # (Note: GANs are highly sensitive; Adam with beta1=0.5 is standard practice)
    lr = 0.0002
    optimizer_G = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = torch.optim.Adam(disc.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    # Mock clean dataset inputs: Normalized between -1.0 and 1.0 (to match Generator Tanh output)
    mock_real_batch = (torch.rand(BATCH_SIZE, *IMG_SHAPE) * 2) - 1

    print("Beginning Verification Pass...")
    print(f"Mock Input Tensor Batch Shape: {mock_real_batch.shape}")

    # Execute step
    d_loss_val, g_loss_val = train_gan_step(
        generator=gen,
        discriminator=disc,
        real_images=mock_real_batch,
        latent_dim=LATENT_DIM,
        g_optimizer=optimizer_G,
        d_optimizer=optimizer_D,
        loss_fn=criterion,
        device=device,
    )

    print("\n--- Step Execution Success ---")
    print(f"Discriminator Optimization Loss Step: {d_loss_val:.4f}")
    print(f"Generator Optimization Loss Step:     {g_loss_val:.4f}")
    
    
# Change this in your verification block:
gen.eval()  # <--- Add this line right before inference
with torch.no_grad():
    test_noise = torch.randn(1, LATENT_DIM).to(device)
    generated_sample = gen(test_noise)
print(f"Standalone Generated Sample Shape:    {generated_sample.shape}")

Beginning Verification Pass...
Mock Input Tensor Batch Shape: torch.Size([16, 1, 28, 28])

--- Step Execution Success ---
Discriminator Optimization Loss Step: 1.3762
Generator Optimization Loss Step:     0.6882
Standalone Generated Sample Shape:    torch.Size([1, 1, 28, 28])
